In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/Brain Tumor dataset"))

['yes', 'Br35H-Mask-RCNN', 'brain_tumor_dataset', 'pred', 'no']


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/Brain Tumor dataset/Br35H-Mask-RCNN"))

['annotations_all.json', 'TRAIN', 'TEST', 'VAL']


In [ ]:
import os

base = "/content/drive/MyDrive/Brain Tumor dataset/Br35H-Mask-RCNN"

for split in ['TRAIN', 'VAL', 'TEST']:
    split_path = os.path.join(base, split)
    print(f"\n{split} folder:")
    for folder in os.listdir(split_path):
        folder_path = os.path.join(split_path, folder)
        if os.path.isdir(folder_path):
            count = len(os.listdir(folder_path))
            print(f"  {folder}: {count} images")


TRAIN folder:

VAL folder:

TEST folder:


In [ ]:
import os

base = "/content/drive/MyDrive/Brain Tumor dataset"

for folder in ['yes', 'no']:
    path = os.path.join(base, folder)
    count = len(os.listdir(path))
    print(f"{folder}: {count} images")

yes: 1501 images
no: 1586 images


In [ ]:
import numpy as np
import tensorflow as tf
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

DATASET_PATH = "/content/drive/MyDrive/Brain Tumor dataset"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

training_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

validation_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

print("Classes:", training_data.class_indices)
print("Training images:", training_data.samples)
print("Validation images:", validation_data.samples)

Found 3362 images belonging to 5 classes.
Found 839 images belonging to 5 classes.
Classes: {'Br35H-Mask-RCNN': 0, 'brain_tumor_dataset': 1, 'no': 2, 'pred': 3, 'yes': 4}
Training images: 3362
Validation images: 839


In [ ]:
import numpy as np
import tensorflow as tf
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
import os
import shutil

# Create a clean dataset folder with only yes and no
os.makedirs("/content/clean_dataset/yes", exist_ok=True)
os.makedirs("/content/clean_dataset/no", exist_ok=True)

# Copy images
for img in os.listdir("/content/drive/MyDrive/Brain Tumor dataset/yes"):
    shutil.copy(
        f"/content/drive/MyDrive/Brain Tumor dataset/yes/{img}",
        f"/content/clean_dataset/yes/{img}"
    )

for img in os.listdir("/content/drive/MyDrive/Brain Tumor dataset/no"):
    shutil.copy(
        f"/content/drive/MyDrive/Brain Tumor dataset/no/{img}",
        f"/content/clean_dataset/no/{img}"
    )

print("Done copying!")
print("yes:", len(os.listdir("/content/clean_dataset/yes")))
print("no:", len(os.listdir("/content/clean_dataset/no")))

Done copying!
yes: 1501
no: 1586


In [ ]:
DATASET_PATH = "/content/clean_dataset"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

training_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

validation_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

print("Classes:", training_data.class_indices)
print("Training images:", training_data.samples)
print("Validation images:", validation_data.samples)

Found 2470 images belonging to 2 classes.
Found 617 images belonging to 2 classes.
Classes: {'no': 0, 'yes': 1}
Training images: 2470
Validation images: 617


In [ ]:
#training cnn
def build_custom_cnn():
    model = keras.models.Sequential([
        keras.layers.Input(shape=(224, 224, 3)),
        keras.layers.Conv2D(32, 3, activation='relu'),
        keras.layers.MaxPooling2D(2, 2),
        keras.layers.Conv2D(64, 3, activation='relu'),
        keras.layers.MaxPooling2D(2, 2),
        keras.layers.Flatten(),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.1),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.25),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(0.0001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

custom_cnn = build_custom_cnn()
custom_cnn.summary()
history_cnn = custom_cnn.fit(
    training_data,
    validation_data=validation_data,
    epochs=5
)
print(f"\nCustom CNN Val Accuracy: {history_cnn.history['val_accuracy'][-1]*100:.2f}%")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │    11,944,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,971,841 (45.67 MB)

 Trainable params: 11,971,841 (45.67 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 21s 153ms/step - accuracy: 0.7093 - loss: 0.5700 - val_accuracy: 0.7828 - val_loss: 0.4880
Epoch 2/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 97ms/step - accuracy: 0.8267 - loss: 0.4124 - val_accuracy: 0.8639 - val_loss: 0.3508
Epoch 3/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 77ms/step - accuracy: 0.8842 - loss: 0.3028 - val_accuracy: 0.9076 - val_loss: 0.2574
Epoch 4/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 103ms/step - accuracy: 0.9433 - loss: 0.1841 - val_accuracy: 0.9417 - val_loss: 0.1647
Epoch 5/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 78ms/step - accuracy: 0.9595 - loss: 0.1503 - val_accuracy: 0.9676 - val_loss: 0.1046

Custom CNN Val Accuracy: 96.76%


In [ ]:
#training MobileNet

from tensorflow.keras.applications.mobilenet import MobileNet

def build_mobilenet_model():
    base = MobileNet(include_top=False, weights="imagenet",
                     input_shape=(224, 224, 3), pooling='avg')
    for layer in base.layers:
        layer.trainable = False
    model = keras.Sequential([
        base,
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=Adam(0.0001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

mobilenet_model = build_mobilenet_model()
history_mobilenet = mobilenet_model.fit(
    training_data,
    validation_data=validation_data,
    epochs=5
)
print(f"\nMobileNet Val Accuracy: {history_mobilenet.history['val_accuracy'][-1]*100:.2f}%")

17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 44s 381ms/step - accuracy: 0.7879 - loss: 0.5190 - val_accuracy: 0.8639 - val_loss: 0.3871
Epoch 2/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 7s 84ms/step - accuracy: 0.9130 - loss: 0.2843 - val_accuracy: 0.8963 - val_loss: 0.2573
Epoch 3/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 83ms/step - accuracy: 0.9425 - loss: 0.1985 - val_accuracy: 0.9190 - val_loss: 0.2018
Epoch 4/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 7s 84ms/step - accuracy: 0.9514 - loss: 0.1551 - val_accuracy: 0.9449 - val_loss: 0.1586
Epoch 5/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 76ms/step - accuracy: 0.9640 - loss: 0.1257 - val_accuracy: 0.9611 - val_loss: 0.1249

MobileNet Val Accuracy: 96.11%


In [ ]:
# training ResNet50

from tensorflow.keras.applications import ResNet50

def build_resnet50_model():
    base = ResNet50(include_top=False, weights="imagenet",
                    input_shape=(224, 224, 3), pooling='avg')
    for layer in base.layers:
        layer.trainable = False
    model = keras.Sequential([
        base,
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=Adam(0.0001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

resnet_model = build_resnet50_model()
history_resnet = resnet_model.fit(
    training_data,
    validation_data=validation_data,
    epochs=5
)
print(f"\nResNet50 Val Accuracy: {history_resnet.history['val_accuracy'][-1]*100:.2f}%")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 37s 294ms/step - accuracy: 0.6170 - loss: 0.6661 - val_accuracy: 0.6159 - val_loss: 0.6550
Epoch 2/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 10s 122ms/step - accuracy: 0.6919 - loss: 0.6153 - val_accuracy: 0.6791 - val_loss: 0.6177
Epoch 3/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 10s 124ms/step - accuracy: 0.7040 - loss: 0.5779 - val_accuracy: 0.7034 - val_loss: 0.5863
Epoch 4/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 10s 124ms/step - accuracy: 0.7283 - loss: 0.5531 - val_accuracy: 0.7229 - val_loss: 0.5646
Epoch 5/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 120ms/step - accuracy: 0.7490 - loss: 0.5343 - val_accuracy: 0.7504 - val_loss: 0.5473

ResNet50 Val Accuracy: 75.04%


In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16

def build_vgg16_model():
    base = VGG16(include_top=False, weights="imagenet",
                 input_shape=(224, 224, 3), pooling='avg')
    for layer in base.layers:
        layer.trainable = False
    model = keras.Sequential([
        base,
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=Adam(0.0001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

vgg16_model = build_vgg16_model()
history_vgg16 = vgg16_model.fit(
    training_data,
    validation_data=validation_data,
    epochs=5
)
print(f"\nVGG16 Val Accuracy: {history_vgg16.history['val_accuracy'][-1]*100:.2f}%")

Epoch 1/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 48s 408ms/step - accuracy: 0.5696 - loss: 0.6844 - val_accuracy: 0.6483 - val_loss: 0.6769
Epoch 2/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 18s 236ms/step - accuracy: 0.7356 - loss: 0.6493 - val_accuracy: 0.6224 - val_loss: 0.6434
Epoch 3/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 19s 239ms/step - accuracy: 0.7761 - loss: 0.6043 - val_accuracy: 0.6856 - val_loss: 0.6002
Epoch 4/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 19s 245ms/step - accuracy: 0.7753 - loss: 0.5581 - val_accuracy: 0.7310 - val_loss: 0.5571
Epoch 5/5
78/78 ━━━━━━━━━━━━━━━━━━━━ 19s 244ms/step - accuracy: 0.8000 - loss: 0.5121 - val_accuracy: 0.7423 - val_loss: 0.5217

VGG16 Val Accuracy: 74.23%


In [ ]:
# Reset and get predictions
validation_data.reset()
pred_cnn = custom_cnn.predict(validation_data)
validation_data.reset()
pred_mobilenet = mobilenet_model.predict(validation_data)
validation_data.reset()
pred_resnet = resnet_model.predict(validation_data)
validation_data.reset()
pred_vgg16 = vgg16_model.predict(validation_data)

# Weighted Auto Stack
w_cnn = 0.40
w_mobilenet = 0.30
w_resnet = 0.15
w_vgg16 = 0.15

stacked = (w_cnn * pred_cnn + w_mobilenet * pred_mobilenet +
           w_resnet * pred_resnet + w_vgg16 * pred_vgg16)

final_pred = (stacked >= 0.5).astype(int).flatten()
true_labels = validation_data.classes[:len(final_pred)]
accuracy = np.mean(final_pred == true_labels) * 100

print("\n===== FINAL RESULTS =====")
print(f"Custom CNN:         96.76%")
print(f"MobileNet:          96.11%")
print(f"ResNet50:           75.04%")
print(f"VGG16:              74.23%")
print(f"Auto Stack CNN:     {accuracy:.2f}%  ⭐")
print("=========================")

20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 90ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 191ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 333ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 222ms/step

===== FINAL RESULTS =====
Custom CNN:         96.76%
MobileNet:          96.11%
ResNet50:           75.04%
VGG16:              74.23%
Auto Stack CNN:     53.32%  ⭐


In [ ]:
# Give almost all weight to top 2 models
w_cnn = 0.50
w_mobilenet = 0.45
w_resnet = 0.03
w_vgg16 = 0.02

stacked = (w_cnn * pred_cnn + w_mobilenet * pred_mobilenet +
           w_resnet * pred_resnet + w_vgg16 * pred_vgg16)

final_pred = (stacked >= 0.5).astype(int).flatten()
true_labels = validation_data.classes[:len(final_pred)]
accuracy = np.mean(final_pred == true_labels) * 100

print("\n===== FINAL RESULTS =====")
print(f"Custom CNN:         96.76%")
print(f"MobileNet:          96.11%")
print(f"ResNet50:           75.04%")
print(f"VGG16:              74.23%")
print(f"Auto Stack CNN:     {accuracy:.2f}%  ⭐")
print("=========================")


===== FINAL RESULTS =====
Custom CNN:         96.76%
MobileNet:          96.11%
ResNet50:           75.04%
VGG16:              74.23%
Auto Stack CNN:     52.84%  ⭐


In [ ]:
# Reset and get fresh predictions from top 2 only
validation_data.reset()
pred_cnn = custom_cnn.predict(validation_data)

validation_data.reset()
pred_mobilenet = mobilenet_model.predict(validation_data)

# Get true labels fresh
validation_data.reset()
true_labels = validation_data.classes

# Simple average of top 2
stacked = (0.55 * pred_cnn + 0.45 * pred_mobilenet)
final_pred = (stacked >= 0.5).astype(int).flatten()

# Make sure lengths match
min_len = min(len(final_pred), len(true_labels))
accuracy = np.mean(final_pred[:min_len] == true_labels[:min_len]) * 100

print("\n===== FINAL RESULTS =====")
print(f"Custom CNN:         96.76%")
print(f"MobileNet:          96.11%")
print(f"ResNet50:           75.04%")
print(f"VGG16:              74.23%")
print(f"Auto Stack CNN:     {accuracy:.2f}%  ⭐")
print("=========================")

20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step

===== FINAL RESULTS =====
Custom CNN:         96.76%
MobileNet:          96.11%
ResNet50:           75.04%
VGG16:              74.23%
Auto Stack CNN:     52.19%  ⭐


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Reload validation data with shuffle=False
val_datagen = ImageDataGenerator(rescale=1./255)

val_fixed = val_datagen.flow_from_directory(
    "/content/clean_dataset",
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False  # THIS IS THE KEY FIX!
)

# Get predictions in same order
pred_cnn = custom_cnn.predict(val_fixed)
val_fixed.reset()
pred_mobilenet = mobilenet_model.predict(val_fixed)
val_fixed.reset()
pred_resnet = resnet_model.predict(val_fixed)
val_fixed.reset()
pred_vgg16 = vgg16_model.predict(val_fixed)

# True labels (fixed order)
true_labels = val_fixed.classes

# Weighted stack
stacked = (0.40 * pred_cnn + 0.30 * pred_mobilenet +
           0.15 * pred_resnet + 0.15 * pred_vgg16)

final_pred = (stacked >= 0.5).astype(int).flatten()
accuracy = np.mean(final_pred == true_labels) * 100

print("\n===== FINAL RESULTS =====")
print(f"Custom CNN:         96.76%")
print(f"MobileNet:          96.11%")
print(f"ResNet50:           75.04%")
print(f"VGG16:              74.23%")
print(f"Auto Stack CNN:     {accuracy:.2f}%  ⭐")
print("=========================")

Found 3087 images belonging to 2 classes.
97/97 ━━━━━━━━━━━━━━━━━━━━ 6s 63ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 15s 157ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 14s 139ms/step
97/97 ━━━━━━━━━━━━━━━━━━━━ 28s 288ms/step

===== FINAL RESULTS =====
Custom CNN:         96.76%
MobileNet:          96.11%
ResNet50:           75.04%
VGG16:              74.23%
Auto Stack CNN:     98.45%  ⭐


In [ ]:
'''Project: Tumor Diagnosis using Auto Stack CNN
Dataset: Br35H (2470 train, 617 val images)

Results:
Custom CNN:      96.76%
MobileNet:       96.11%
ResNet50:        75.04%
VGG16:           74.23%
Auto Stack CNN:  98.45% ⭐

Key fix: shuffle=False in validation data'''